# Circuit Transformation

DeepLog circuits can be **transformed** from one algebraic structure to another. This is the mechanism behind the `expectation` aggregation operator, which transforms boolean proof circuits into the probability semiring.

This notebook demonstrates the three levels of the transformation API:
1. `transform_circuit()` — low-level function operating on raw circuits and node IDs
2. `transform_nodes()` — batch-transforms multiple `CircuitNode` objects
3. `CircuitNode.transform_circuit()` — per-node convenience method

## Building a boolean circuit

We start by constructing a small boolean circuit: `(a OR b) AND (NOT a OR c)`.

In [ ]:
import torch
from deeplog.circuit import Circuit, CircuitNode, to_module, transform_circuit, transform_nodes
from deeplog import BOOLEAN, PROBABILITY

bool_circuit = Circuit("boolean")
a = bool_circuit.get_leaf_node(("a",))
b = bool_circuit.get_leaf_node(("b",))
c = bool_circuit.get_leaf_node(("c",))

or_op = bool_circuit.get_operator("or")
and_op = bool_circuit.get_operator("and")
not_op = bool_circuit.get_operator("not")

# (a OR b)
clause1 = or_op(a, b)
# (NOT a OR c)
clause2 = or_op(not_op(a), c)
# (a OR b) AND (NOT a OR c)
root = and_op(clause1, clause2)

# Evaluate as boolean
bool_module = bool_circuit.to_module({root: ("result",)})
# a=1, b=0, c=1 -> (1 OR 0) AND (NOT 1 OR 1) = 1 AND 1 = 1
print("Boolean result:", bool_module(torch.tensor([[1.0, 0.0, 1.0]])))

## Basic transformation with `transform_circuit()`

The `transform_circuit()` function converts a circuit to a different algebraic structure. When both structures are `Semiring` or `Algebra` instances, operator mapping is inferred automatically:
- `and` → `times`
- `or` → `plus`
- `not` → `negate`

In [ ]:
prob_circuit, node_map = transform_circuit(
    bool_circuit, "probability", roots=[root]
)

print("Source structure:", bool_circuit.structure)
print("Target structure:", prob_circuit.structure)

# Build a module from the transformed circuit
prob_module = prob_circuit.to_module({node_map[root]: ("result",)})
# With probabilities: P(a)=0.8, P(b)=0.3, P(c)=0.6
# P((a OR b) AND (NOT a OR c)) = 0.54
probs = torch.tensor([[0.8, 0.3, 0.6]])
print("Probability result:", prob_module(probs))

## Leaf remapping

Use `leaf_mapping` to rename symbols during transformation. This is useful when you need to distinguish boolean atoms from their probability counterparts.

In [ ]:
def bool_to_prob_symbol(symbol):
    """Tag boolean leaf symbols with the probability structure."""
    return ("_", symbol, ("probability",))

mapped_circuit, mapped_nodes = transform_circuit(
    bool_circuit, "probability", roots=[root],
    leaf_mapping=bool_to_prob_symbol,
)

mapped_module = mapped_circuit.to_module({mapped_nodes[root]: ("result",)})

# Input symbols are now tagged: ('_', ('a',), ('probability',)), etc.
print("Input shape:", mapped_module.get_input_shape())
print("Result:", mapped_module(probs))

## Batch transformation with `transform_nodes()`

`transform_nodes()` transforms multiple `CircuitNode` objects from the same circuit in a single pass. This is more efficient than transforming each node individually because shared subgraphs are only traversed once.

In [ ]:
# Wrap the two clause roots as CircuitNodes
node_clause1 = CircuitNode(bool_circuit, clause1)
node_clause2 = CircuitNode(bool_circuit, clause2)

# Transform both in one pass
transformed = transform_nodes(
    node_clause1, node_clause2,
    target_structure="probability",
)

print(f"Transformed {len(transformed)} nodes")
print("Both share the same circuit:", transformed[0].circuit is transformed[1].circuit)

# Convert to a module with two outputs
multi_module = to_module(
    *transformed,
    names=(("clause1",), ("clause2",)),
)
print("Output shape:", multi_module.get_output_shape())
print("Results:", multi_module(probs))

## Per-node convenience: `CircuitNode.transform_circuit()`

Individual `CircuitNode` objects also expose a `transform_circuit()` method for quick one-off transformations.

In [ ]:
bool_node = CircuitNode(bool_circuit, root)
print("Original structure:", bool_node.get_structure())

prob_node = bool_node.transform_circuit("probability")
print("Transformed structure:", prob_node.get_structure())

module = prob_node.to_module(name=("result",))
print("Result:", module(probs))

## Automatic operator mapping

Auto-mapping works because the built-in structures (`BOOLEAN`, `PROBABILITY`, `LOGPROBABILITY`) are all `Algebra` instances with named roles. You can inspect these roles directly.

In [ ]:
print(f"BOOLEAN:     product={BOOLEAN.product!r}, sum={BOOLEAN.sum!r}, negation={BOOLEAN.negation!r}")
print(f"PROBABILITY: product={PROBABILITY.product!r}, sum={PROBABILITY.sum!r}, negation={PROBABILITY.negation!r}")
print()
print("Auto-mapping: and->times, or->plus, not->negate")

For custom structures that aren't `Semiring`/`Algebra`, you must provide an explicit `operator_mapping`.

In [ ]:
from deeplog import AlgebraicStructure

fuzzy = AlgebraicStructure(
    name="fuzzy",
    operator_fns={
        "t_norm": lambda a, b: a * b,
        "t_conorm": lambda x, y: x + y - x * y,
        "complement": lambda x: 1.0 - x,
    },
)

fuzzy_circuit, fuzzy_map = transform_circuit(
    bool_circuit, fuzzy, roots=[root],
    operator_mapping={"and": "t_norm", "or": "t_conorm", "not": "complement"},
)

fuzzy_module = fuzzy_circuit.to_module({fuzzy_map[root]: ("result",)})
print("Fuzzy result:", fuzzy_module(probs))